# OSS Health Final Modeling

`new_label`을 최종 target으로 사용한다.

이 노트북의 목적은 다음과 같다.

1. feature engineering 결과를 기반으로 후보 dataset을 만든다.
2. univariate, linear coefficient, tree importance, permutation importance, SHAP을 종합해 feature subset을 탐색한다.
3. 여러 model과 dataset 조합을 cross validation으로 비교한다.
4. 최고 조합에 대해 hyperparameter tuning을 수행한다.
5. 최종 모델, feature list, metadata를 backend inference에 사용할 수 있도록 저장한다.


## Imports and Settings

In [3]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu
from scipy.stats import randint, uniform, loguniform

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.svm import SVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import shap
import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 42
N_JOBS = 1
TARGET_COL = "new_label"

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data/main/dataset.csv"
OUTPUT_DIR = BASE_DIR / "outputs/2_model"
MODEL_DIR = BASE_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


## Load Dataset

In [4]:
if not DATA_PATH.exists():
    DATA_PATH = Path("/Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/data/main/dataset.csv")

df_raw = pd.read_csv(DATA_PATH)

print("Data path:", DATA_PATH)
print("Raw shape:", df_raw.shape)
print("Original label distribution:")
print(df_raw["label"].value_counts())

df_raw.head()


Data path: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/data/main/dataset.csv
Raw shape: (411, 83)
Original label distribution:
label
1    274
0    137
Name: count, dtype: int64


,Unnamed: 0,repo_name,num_contributors,total_contributions,top1_contribution_share,top5_contribution_share,contribution_gini,median_contributions,num_deployments,has_deployments,num_unique_refs,tag_based_deployment_ratio,deployment_recency_days,num_events,num_unique_event_types,dominant_event_type,dominant_event_ratio,event_type_entropy,has_IssuesEvent,has_PullRequestEvent,has_IssueCommentEvent,recent_event_density,IssuesEvent_ratio,IssueCommentEvent_ratio,PullRequestEvent_ratio,PushEvent_ratio,WatchEvent_ratio,ForkEvent_ratio,stargazers_count,subscribers_count,subscribers_to_stars_ratio,primary_language_ratio,top2_ratio,top3_ratio,language_entropy,minor_lang_ratio,infra_ratio,markup_ratio,is_monolingual,compiled_ratio,num_tags,stable_tag_ratio,prerelease_tag_ratio,latest_tag_is_stable,latest_tag_is_prerelease,semver_tag_ratio,num_major_versions,num_minor_versions,repo_age_days,last_update_recency_days,last_push_recency_days,update_push_gap_days,repo_size,forks_count,open_issues_count,network_count,has_issues,has_projects,has_downloads,has_wiki,has_pages,has_discussions,archived,disabled,allow_forking,has_pull_requests,forks_to_stars_ratio,open_issues_to_stars_ratio,stars_per_repo_age_day,forks_per_repo_age_day,top1_contribution_ratio,top3_contribution_ratio,contribution_entropy,contributors_to_stars_ratio,tag_release_velocity,interaction_ratio,development_ratio,external_interest_event_ratio,stars_per_size,forks_per_size,issues_per_size,error,label
0,0,feast-dev/feast,30.0,3139.0,0.173941,0.475948,0.475385,61.5,30.0,1.0,2.0,0.0,24.292122,30.0,8.0,PushEvent,0.366667,1.814285,0.0,1.0,1.0,14.478587,0.000000,0.100000,0.100000,0.366667,0.166667,0.100000,7004.0,85.0,0.012136,0.763338,0.889846,0.939283,0.894262,0.236662,0.012359,0.003585,0.0,0.153934,30.0,1.000000,0.000000,1.0,0.0,1.000000,1.0,24.0,2703.643302,1.189633,1.457353,0.267720,254431.0,1315.0,346.0,1315.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.187750,0.049400,2.590578,0.486381,0.173941,0.339599,3.005782,0.004283,0.011096,0.100000,0.466667,0.266667,0.027528,0.005168,0.001360,NaN,1
1,1,tmux/tmux,19.0,9800.0,0.802143,0.997551,0.923963,2.0,0.0,0.0,0.0,0.0,0.000000,30.0,6.0,IssueCommentEvent,0.466667,1.244199,1.0,1.0,1.0,24.062831,0.033333,0.466667,0.033333,0.033333,0.366667,0.000000,45140.0,470.0,0.010412,0.872271,0.944649,0.972869,0.541481,0.127729,0.030302,0.000000,0.0,0.872271,30.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,3988.970992,0.056640,0.324546,0.267905,19142.0,2603.0,55.0,2603.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.057665,0.001218,11.316202,0.652549,0.802143,0.996735,0.529017,0.000421,0.007521,0.500000,0.066667,0.366667,2.358165,0.135984,0.002873,NaN,1
2,2,ultralytics/ultralytics,30.0,3551.0,0.443537,0.813292,0.779283,21.5,30.0,1.0,1.0,0.0,0.788279,30.0,7.0,IssueCommentEvent,0.333333,1.636432,0.0,1.0,1.0,12.361163,0.000000,0.333333,0.133333,0.233333,0.200000,0.033333,57179.0,256.0,0.004477,0.996246,0.997740,0.999058,0.028770,0.003754,0.002435,0.001319,1.0,0.000000,30.0,1.000000,0.000000,1.0,0.0,1.000000,1.0,1.0,1342.516161,0.064101,0.082318,0.018218,55086.0,11006.0,325.0,11006.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.192483,0.005684,42.590921,8.198039,0.443537,0.666291,2.058244,0.000525,0.022346,0.333333,0.366667,0.233333,1.037995,0.199797,0.005900,NaN,1
3,3,denoland/fresh,30.0,1396.0,0.474928,0.803009,0.767240,8.5,30.0,1.0,30.0,0.0,177.442465,30.0,7.0,IssueCommentEvent,0.233333,1.869789,1.0,1.0,1.0,15.180977,0.100000,0.233333,0.166667,0.200000,0.133333,0.000000,13757.0,78.0,0.005670,0.965551,0.999287,0.999713,0.153832,0.034449,0.000000,0.034162,1.0,0.000000,30.0,1.000000,0.000000,1.0,0.0,1.000000,2.0,7.0,1834.599769,0.015093,0.613854,0.598762,45712.0,749.0,82.0,749.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.054445,0.005961,7.498638,0.408263,0.474928,0.704155,2.023641,0.002181,0.016352,0.333333,0.366667,0.133333,0.300949,0.016385,0.001794,NaN,1
4,4,milvus-io/milvus,30.0,17783.0,0.096947,0.307372,0.252978,506.5,0.0,0.0,0.0,0.0,0.000000,30.0,5.0,PullReque

## Feature Family and Health Score Label

In [5]:
feature_families = {
    "contributor": [
        "num_contributors",
        "total_contributions",
        "top1_contribution_share",
        "top5_contribution_share",
        "contribution_gini",
        "median_contributions",
        "top1_contribution_ratio",
        "top3_contribution_ratio",
        "contribution_entropy",
        "contributors_to_stars_ratio",
    ],
    "release": [
        "num_deployments",
        "has_deployments",
        "num_unique_refs",
        "tag_based_deployment_ratio",
        "deployment_recency_days",
        "num_tags",
        "stable_tag_ratio",
        "prerelease_tag_ratio",
        "latest_tag_is_stable",
        "latest_tag_is_prerelease",
        "semver_tag_ratio",
        "num_major_versions",
        "num_minor_versions",
        "tag_release_velocity",
    ],
    "event": [
        "num_events",
        "num_unique_event_types",
        "dominant_event_ratio",
        "event_type_entropy",
        "has_IssuesEvent",
        "has_PullRequestEvent",
        "has_IssueCommentEvent",
        "recent_event_density",
        "IssuesEvent_ratio",
        "IssueCommentEvent_ratio",
        "PullRequestEvent_ratio",
        "PushEvent_ratio",
        "WatchEvent_ratio",
        "ForkEvent_ratio",
        "interaction_ratio",
        "development_ratio",
        "external_interest_event_ratio",
    ],
    "popularity": [
        "stargazers_count",
        "subscribers_count",
        "subscribers_to_stars_ratio",
        "forks_count",
        "network_count",
        "forks_to_stars_ratio",
        "open_issues_to_stars_ratio",
        "stars_per_repo_age_day",
        "forks_per_repo_age_day",
        "stars_per_size",
        "forks_per_size",
    ],
    "language": [
        "primary_language_ratio",
        "top2_ratio",
        "top3_ratio",
        "language_entropy",
        "minor_lang_ratio",
        "infra_ratio",
        "markup_ratio",
        "is_monolingual",
        "compiled_ratio",
    ],
    "maintenance": [
        "repo_age_days",
        "last_update_recency_days",
        "last_push_recency_days",
        "update_push_gap_days",
    ],
    "governance": [
        "repo_size",
        "open_issues_count",
        "has_issues",
        "has_projects",
        "has_downloads",
        "has_wiki",
        "has_pages",
        "has_discussions",
        "archived",
        "disabled",
        "allow_forking",
        "has_pull_requests",
        "issues_per_size",
    ],
}

negative_features = [
    "top1_contribution_share",
    "top5_contribution_share",
    "contribution_gini",
    "top1_contribution_ratio",
    "top3_contribution_ratio",
    "deployment_recency_days",
    "prerelease_tag_ratio",
    "latest_tag_is_prerelease",
    "last_update_recency_days",
    "last_push_recency_days",
    "update_push_gap_days",
    "open_issues_to_stars_ratio",
    "issues_per_size",
    "dominant_event_ratio",
    "archived",
    "disabled",
]

score_dimensions = {
    "community_activity": [
        "num_events",
        "num_unique_event_types",
        "event_type_entropy",
        "recent_event_density",
        "has_IssuesEvent",
        "has_PullRequestEvent",
        "has_IssueCommentEvent",
        "IssuesEvent_ratio",
        "IssueCommentEvent_ratio",
        "PullRequestEvent_ratio",
        "PushEvent_ratio",
        "interaction_ratio",
        "development_ratio",
    ],
    "contributor_sustainability": feature_families["contributor"],
    "release_engineering": feature_families["release"],
    "popularity_adoption": [
        "stargazers_count",
        "subscribers_count",
        "forks_count",
        "network_count",
        "subscribers_to_stars_ratio",
        "forks_to_stars_ratio",
        "stars_per_repo_age_day",
        "forks_per_repo_age_day",
        "stars_per_size",
        "forks_per_size",
        "external_interest_event_ratio",
    ],
    "language_structure": feature_families["language"],
    "maintenance": feature_families["maintenance"],
    "governance": [
        "repo_size",
        "open_issues_count",
        "has_issues",
        "has_projects",
        "has_downloads",
        "has_wiki",
        "has_pages",
        "has_discussions",
        "archived",
        "disabled",
        "allow_forking",
        "has_pull_requests",
        "open_issues_to_stars_ratio",
        "issues_per_size",
    ],
}


def existing_columns(columns, data):
    return [col for col in columns if col in data.columns]


def to_numeric_columns(data, columns):
    out = data.copy()

    for col in existing_columns(columns, out):
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


def minmax_series(values):
    values = values.astype(float)
    min_value = values.min()
    max_value = values.max()

    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series(0.5, index=values.index)

    return (values - min_value) / (max_value - min_value)


def build_health_score_labels(data):
    out = data.copy()
    dimensions = {
        dim: existing_columns(cols, out)
        for dim, cols in score_dimensions.items()
    }
    score_features = sorted(set(sum(dimensions.values(), [])))

    out = to_numeric_columns(out, score_features)

    scaled = pd.DataFrame(index=out.index)

    for col in score_features:
        filled = out[col].fillna(out[col].median())
        scaled[col] = minmax_series(filled)

        if col in negative_features:
            scaled[col] = 1 - scaled[col]

    for dim, cols in dimensions.items():
        out[f"{dim}_score"] = scaled[cols].mean(axis=1)

    dimension_score_cols = [f"{dim}_score" for dim in dimensions]
    out["oss_health_score"] = out[dimension_score_cols].mean(axis=1)
    out["new_label"] = (
        out["oss_health_score"] >= out["oss_health_score"].median()
    ).astype(int)

    return out, dimension_score_cols


df_labeled, dimension_score_cols = build_health_score_labels(df_raw)

print("new_label distribution:")
print(df_labeled["new_label"].value_counts())
print()
print("original label vs new_label:")
print(pd.crosstab(df_labeled["label"], df_labeled["new_label"]))


new_label distribution:
new_label
1    206
0    205
Name: count, dtype: int64

original label vs new_label:
new_label    0    1
label              
0          111   26
1           94  180


## Feature Engineering

In [6]:
def add_engineered_features(data):
    out = data.copy()
    eps = 1e-9
    raw_feature_cols = sorted(set(sum(feature_families.values(), [])))

    out = to_numeric_columns(out, raw_feature_cols)

    out["is_recently_pushed_30d"] = (out["last_push_recency_days"] <= 30).astype(int)
    out["is_recently_updated_90d"] = (out["last_update_recency_days"] <= 90).astype(int)
    out["is_recently_pushed_90d"] = (out["last_push_recency_days"] <= 90).astype(int)
    out["is_stale_365d"] = (out["last_push_recency_days"] > 365).astype(int)

    out["push_update_consistency"] = 1 / (1 + out["update_push_gap_days"])
    out["freshness_score"] = (
        1 / (1 + out["last_push_recency_days"])
        + 1 / (1 + out["last_update_recency_days"])
    ) / 2

    out["bus_factor_risk"] = (
        out["top1_contribution_share"] + out["contribution_gini"]
    ) / 2
    out["distributed_contribution_score"] = (
        out["contribution_entropy"] * (1 - out["top1_contribution_share"])
    )
    out["contributor_depth_score"] = (
        np.log1p(out["num_contributors"])
        * np.log1p(out["median_contributions"])
    )

    out["release_maturity_score"] = (
        out["stable_tag_ratio"]
        * out["semver_tag_ratio"]
        * out["latest_tag_is_stable"]
    )
    out["active_release_score"] = (
        np.log1p(out["num_tags"]) * out["tag_release_velocity"]
    )
    out["release_recency_score"] = 1 / (1 + out["deployment_recency_days"])
    out["release_quality_score"] = (
        out["release_maturity_score"]
        + out["release_recency_score"]
        + minmax_series(out["active_release_score"].fillna(0))
    ) / 3

    out["collaboration_event_score"] = (
        out["PullRequestEvent_ratio"]
        + out["IssueCommentEvent_ratio"]
        + out["IssuesEvent_ratio"]
    )
    out["activity_diversity_score"] = (
        out["event_type_entropy"] * out["num_unique_event_types"]
    )
    out["healthy_activity_score"] = (
        out["interaction_ratio"]
        + out["development_ratio"]
        + out["collaboration_event_score"]
    ) / 3
    out["non_external_activity_ratio"] = 1 - out["external_interest_event_ratio"]

    out["adoption_efficiency"] = (
        np.log1p(out["stargazers_count"])
        / np.log1p(out["repo_age_days"] + 1)
    )
    out["issue_burden_score"] = (
        out["open_issues_count"]
        / (np.log1p(out["stargazers_count"]) + eps)
    )
    out["fork_interest_efficiency"] = (
        np.log1p(out["forks_count"])
        / np.log1p(out["repo_age_days"] + 1)
    )

    out["governance_openness_score"] = (
        out["has_issues"]
        + out["has_projects"]
        + out["has_wiki"]
        + out["has_discussions"]
        + out["has_pull_requests"]
    ) / 5
    out["negative_repo_state"] = (
        out["archived"] + out["disabled"]
    ).clip(0, 1)

    out["maintainer_activity_score"] = (
        out["is_recently_pushed_30d"]
        + out["is_recently_updated_90d"]
        + out["push_update_consistency"]
    ) / 3

    out.replace([np.inf, -np.inf], np.nan, inplace=True)

    return out


df_fe = add_engineered_features(df_labeled)


## Modeling Feature Set

In [7]:
candidate_raw_features = [
    "num_contributors",
    "total_contributions",
    "top1_contribution_share",
    "top5_contribution_share",
    "contribution_gini",
    "median_contributions",
    "top1_contribution_ratio",
    "top3_contribution_ratio",
    "contribution_entropy",
    "contributors_to_stars_ratio",
    "num_deployments",
    "has_deployments",
    "num_unique_refs",
    "tag_based_deployment_ratio",
    "deployment_recency_days",
    "num_tags",
    "stable_tag_ratio",
    "prerelease_tag_ratio",
    "latest_tag_is_stable",
    "latest_tag_is_prerelease",
    "semver_tag_ratio",
    "num_major_versions",
    "num_minor_versions",
    "tag_release_velocity",
    "num_events",
    "num_unique_event_types",
    "dominant_event_ratio",
    "event_type_entropy",
    "has_IssuesEvent",
    "has_PullRequestEvent",
    "has_IssueCommentEvent",
    "recent_event_density",
    "IssuesEvent_ratio",
    "IssueCommentEvent_ratio",
    "PullRequestEvent_ratio",
    "PushEvent_ratio",
    "WatchEvent_ratio",
    "ForkEvent_ratio",
    "interaction_ratio",
    "development_ratio",
    "external_interest_event_ratio",
    "stargazers_count",
    "subscribers_count",
    "subscribers_to_stars_ratio",
    "forks_count",
    "network_count",
    "forks_to_stars_ratio",
    "open_issues_to_stars_ratio",
    "stars_per_repo_age_day",
    "forks_per_repo_age_day",
    "stars_per_size",
    "forks_per_size",
    "primary_language_ratio",
    "top2_ratio",
    "top3_ratio",
    "language_entropy",
    "minor_lang_ratio",
    "infra_ratio",
    "markup_ratio",
    "is_monolingual",
    "compiled_ratio",
    "repo_age_days",
    "last_update_recency_days",
    "last_push_recency_days",
    "update_push_gap_days",
    "repo_size",
    "open_issues_count",
    "has_issues",
    "has_projects",
    "has_downloads",
    "has_wiki",
    "has_pages",
    "has_discussions",
    "archived",
    "disabled",
    "allow_forking",
    "has_pull_requests",
    "issues_per_size",
]

engineered_features = [
    "is_recently_pushed_30d",
    "is_recently_updated_90d",
    "is_recently_pushed_90d",
    "is_stale_365d",
    "push_update_consistency",
    "freshness_score",
    "bus_factor_risk",
    "distributed_contribution_score",
    "contributor_depth_score",
    "release_maturity_score",
    "active_release_score",
    "release_recency_score",
    "release_quality_score",
    "collaboration_event_score",
    "activity_diversity_score",
    "healthy_activity_score",
    "non_external_activity_ratio",
    "adoption_efficiency",
    "issue_burden_score",
    "fork_interest_efficiency",
    "governance_openness_score",
    "negative_repo_state",
    "maintainer_activity_score",
]

leakage_columns = [
    "label",
    "new_label",
    "oss_health_score",
    "dominant_event_type",
    "error",
    "repo_name",
    "Unnamed: 0",
] + dimension_score_cols

all_model_features = existing_columns(candidate_raw_features + engineered_features, df_fe)
all_model_features = [col for col in all_model_features if col not in leakage_columns]

modeling_df = df_fe[all_model_features + [TARGET_COL]].copy()

for col in all_model_features:
    modeling_df[col] = pd.to_numeric(modeling_df[col], errors="coerce")

print("Modeling shape:", modeling_df.shape)
print("Number of model features:", len(all_model_features))
print("Target distribution:")
print(modeling_df[TARGET_COL].value_counts())


Modeling shape: (411, 102)
Number of model features: 101
Target distribution:
new_label
1    206
0    205
Name: count, dtype: int64


In [8]:
feature_family_map = {
    "contributor": [
        "num_contributors",
        "total_contributions",
        "top1_contribution_share",
        "top5_contribution_share",
        "contribution_gini",
        "median_contributions",
        "top1_contribution_ratio",
        "top3_contribution_ratio",
        "contribution_entropy",
        "contributors_to_stars_ratio",
        "bus_factor_risk",
        "distributed_contribution_score",
        "contributor_depth_score",
    ],
    "release": [
        "num_deployments",
        "has_deployments",
        "num_unique_refs",
        "tag_based_deployment_ratio",
        "deployment_recency_days",
        "num_tags",
        "stable_tag_ratio",
        "prerelease_tag_ratio",
        "latest_tag_is_stable",
        "latest_tag_is_prerelease",
        "semver_tag_ratio",
        "num_major_versions",
        "num_minor_versions",
        "tag_release_velocity",
        "release_maturity_score",
        "active_release_score",
        "release_recency_score",
        "release_quality_score",
    ],
    "event": [
        "num_events",
        "num_unique_event_types",
        "dominant_event_ratio",
        "event_type_entropy",
        "has_IssuesEvent",
        "has_PullRequestEvent",
        "has_IssueCommentEvent",
        "recent_event_density",
        "IssuesEvent_ratio",
        "IssueCommentEvent_ratio",
        "PullRequestEvent_ratio",
        "PushEvent_ratio",
        "WatchEvent_ratio",
        "ForkEvent_ratio",
        "interaction_ratio",
        "development_ratio",
        "external_interest_event_ratio",
        "collaboration_event_score",
        "activity_diversity_score",
        "healthy_activity_score",
        "non_external_activity_ratio",
    ],
    "popularity": [
        "stargazers_count",
        "subscribers_count",
        "subscribers_to_stars_ratio",
        "forks_count",
        "network_count",
        "forks_to_stars_ratio",
        "open_issues_to_stars_ratio",
        "stars_per_repo_age_day",
        "forks_per_repo_age_day",
        "stars_per_size",
        "forks_per_size",
        "adoption_efficiency",
        "issue_burden_score",
        "fork_interest_efficiency",
    ],
    "language": [
        "primary_language_ratio",
        "top2_ratio",
        "top3_ratio",
        "language_entropy",
        "minor_lang_ratio",
        "infra_ratio",
        "markup_ratio",
        "is_monolingual",
        "compiled_ratio",
    ],
    "maintenance": [
        "repo_age_days",
        "last_update_recency_days",
        "last_push_recency_days",
        "update_push_gap_days",
        "is_recently_pushed_30d",
        "is_recently_updated_90d",
        "is_recently_pushed_90d",
        "is_stale_365d",
        "push_update_consistency",
        "freshness_score",
        "maintainer_activity_score",
    ],
    "governance": [
        "repo_size",
        "open_issues_count",
        "has_issues",
        "has_projects",
        "has_downloads",
        "has_wiki",
        "has_pages",
        "has_discussions",
        "archived",
        "disabled",
        "allow_forking",
        "has_pull_requests",
        "issues_per_size",
        "governance_openness_score",
        "negative_repo_state",
    ],
}

feature_to_family = {
    feature: family
    for family, features in feature_family_map.items()
    for feature in features
}

feature_catalog = pd.DataFrame({
    "feature": all_model_features,
    "family": [feature_to_family.get(feature, "other") for feature in all_model_features],
    "source": ["engineered" if feature in engineered_features else "raw" for feature in all_model_features],
})

feature_catalog.to_csv(OUTPUT_DIR / "feature_catalog.csv", index=False)
feature_catalog


,feature,family,source
0,num_contributors,contributor,raw
1,total_contributions,contributor,raw
2,top1_contribution_share,contributor,raw
3,top5_contribution_share,contributor,raw
4,contribution_gini,contributor,raw
5,median_contributions,contributor,raw
6,top1_contribution_ratio,contributor,raw
7,top3_contribution_ratio,contributor,raw
8,contribution_entropy,contributor,raw
9,contributors_to_stars_ratio,contributor,raw


## Train Test Split

In [9]:
X_all = modeling_df[all_model_features].copy()
y = modeling_df[TARGET_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:")
print(y_train.value_counts(normalize=True))


X_train: (328, 101)
X_test : (83, 101)
y_train:
new_label
0    0.5
1    0.5
Name: proportion, dtype: float64


## Feature Signal Analysis

In [10]:
def univariate_analysis(X, y):
    rows = []

    for feature in X.columns:
        x0 = X.loc[y == 0, feature].dropna()
        x1 = X.loc[y == 1, feature].dropna()

        mean0 = x0.mean()
        mean1 = x1.mean()
        median0 = x0.median()
        median1 = x1.median()
        pooled_std = np.sqrt((x0.var(ddof=1) + x1.var(ddof=1)) / 2)
        cohen_d = (mean1 - mean0) / pooled_std if pooled_std > 0 else np.nan

        try:
            p_value = mannwhitneyu(x0, x1, alternative="two-sided").pvalue
        except ValueError:
            p_value = np.nan

        rows.append({
            "feature": feature,
            "family": feature_to_family.get(feature, "other"),
            "source": "engineered" if feature in engineered_features else "raw",
            "unhealthy_mean": mean0,
            "healthy_mean": mean1,
            "unhealthy_median": median0,
            "healthy_median": median1,
            "mean_diff": mean1 - mean0,
            "median_diff": median1 - median0,
            "cohen_d": cohen_d,
            "abs_cohen_d": abs(cohen_d),
            "p_value": p_value,
        })

    return pd.DataFrame(rows).sort_values(
        ["abs_cohen_d", "p_value"],
        ascending=[False, True],
    ).reset_index(drop=True)


def get_linear_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=20000,
            solver="liblinear",
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ])


def linear_odds_analysis(X, y):
    model = get_linear_pipeline()
    model.fit(X, y)
    coef = model.named_steps["model"].coef_[0]

    return pd.DataFrame({
        "feature": X.columns,
        "family": [feature_to_family.get(feature, "other") for feature in X.columns],
        "coef": coef,
        "odds_ratio_per_1sd": np.exp(coef),
        "abs_coef": np.abs(coef),
    }).sort_values("abs_coef", ascending=False).reset_index(drop=True)


def get_tree_importance_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=800,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )),
    ])


def tree_importance_analysis(X, y):
    model = get_tree_importance_model()
    model.fit(X, y)
    importance = model.named_steps["model"].feature_importances_

    return pd.DataFrame({
        "feature": X.columns,
        "family": [feature_to_family.get(feature, "other") for feature in X.columns],
        "importance": importance,
    }).sort_values("importance", ascending=False).reset_index(drop=True), model


def permutation_importance_analysis(model, X, y):
    result = permutation_importance(
        model,
        X,
        y,
        n_repeats=20,
        random_state=RANDOM_STATE,
        scoring="roc_auc",
        n_jobs=N_JOBS,
    )

    return pd.DataFrame({
        "feature": X.columns,
        "family": [feature_to_family.get(feature, "other") for feature in X.columns],
        "permutation_importance_mean": result.importances_mean,
        "permutation_importance_std": result.importances_std,
    }).sort_values("permutation_importance_mean", ascending=False).reset_index(drop=True)


def shap_importance_analysis(model, X):
    imputer = model.named_steps["imputer"]
    tree_model = model.named_steps["model"]
    X_imputed = imputer.transform(X)

    explainer = shap.TreeExplainer(tree_model)
    shap_values = explainer.shap_values(X_imputed)

    if isinstance(shap_values, list):
        shap_for_positive_class = np.asarray(shap_values[1])
    else:
        shap_array = np.asarray(shap_values)

        if shap_array.ndim == 3 and shap_array.shape[2] == 2:
            shap_for_positive_class = shap_array[:, :, 1]
        elif shap_array.ndim == 3 and shap_array.shape[0] == 2:
            shap_for_positive_class = shap_array[1, :, :]
        elif shap_array.ndim == 2:
            shap_for_positive_class = shap_array
        else:
            raise ValueError(f"Unexpected SHAP shape: {shap_array.shape}")

    mean_abs_shap = np.abs(shap_for_positive_class).mean(axis=0).ravel()

    return pd.DataFrame({
        "feature": X.columns,
        "family": [feature_to_family.get(feature, "other") for feature in X.columns],
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)


In [11]:
univariate_result = univariate_analysis(X_train, y_train)
linear_result = linear_odds_analysis(X_train, y_train)
tree_result, tree_importance_model = tree_importance_analysis(X_train, y_train)
permutation_result = permutation_importance_analysis(tree_importance_model, X_test, y_test)
shap_result = shap_importance_analysis(tree_importance_model, X_test)

univariate_result.to_csv(OUTPUT_DIR / "univariate_result.csv", index=False)
linear_result.to_csv(OUTPUT_DIR / "linear_result.csv", index=False)
tree_result.to_csv(OUTPUT_DIR / "tree_importance_result.csv", index=False)
permutation_result.to_csv(OUTPUT_DIR / "permutation_importance_result.csv", index=False)
shap_result.to_csv(OUTPUT_DIR / "shap_importance_result.csv", index=False)

shap_result.head(30)


,feature,family,mean_abs_shap
0,has_IssueCommentEvent,event,0.039748
1,is_recently_pushed_30d,maintenance,0.029718
2,has_PullRequestEvent,event,0.026531
3,push_update_consistency,maintenance,0.019979
4,contribution_entropy,contributor,0.019597
5,has_discussions,governance,0.018557
6,latest_tag_is_stable,release,0.017800
7,distributed_contribution_score,contributor,0.017693
8,is_recently_pushed_90d,maintenance,0.017486
9,semver_tag_ratio,release,0.016851


## Family Ablation

In [12]:
def score_feature_set_cv(features, estimator=None):
    if estimator is None:
        estimator = get_tree_importance_model()

    scores = cross_validate(
        estimator,
        X_train[features],
        y_train,
        cv=cv,
        scoring={
            "roc_auc": "roc_auc",
            "f1": "f1",
            "accuracy": "accuracy",
        },
        n_jobs=N_JOBS,
        return_train_score=False,
    )

    return {
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std(),
        "f1_mean": scores["test_f1"].mean(),
        "accuracy_mean": scores["test_accuracy"].mean(),
    }


def family_ablation(features):
    rows = []
    families = sorted(set(feature_to_family.get(feature, "other") for feature in features))

    all_score = score_feature_set_cv(features)
    rows.append({
        "setting": "all_features",
        "family": "all",
        "num_features": len(features),
        **all_score,
    })

    for family in families:
        family_features = [
            feature for feature in features
            if feature_to_family.get(feature, "other") == family
        ]

        if len(family_features) == 0:
            continue

        score = score_feature_set_cv(family_features)
        rows.append({
            "setting": f"only_{family}",
            "family": family,
            "num_features": len(family_features),
            **score,
        })

    for family in families:
        remaining_features = [
            feature for feature in features
            if feature_to_family.get(feature, "other") != family
        ]

        if len(remaining_features) == 0:
            continue

        score = score_feature_set_cv(remaining_features)
        rows.append({
            "setting": f"without_{family}",
            "family": family,
            "num_features": len(remaining_features),
            **score,
        })

    return pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)


ablation_result = family_ablation(all_model_features)
ablation_result.to_csv(OUTPUT_DIR / "family_ablation_result.csv", index=False)
ablation_result


,setting,family,num_features,roc_auc_mean,roc_auc_std,f1_mean,accuracy_mean
0,without_maintenance,maintenance,90,0.977146,0.013517,0.918238,0.914872
1,without_event,event,80,0.976624,0.023217,0.908254,0.905828
2,without_popularity,popularity,87,0.975121,0.018088,0.911471,0.908671
3,all_features,all,101,0.974409,0.020342,0.909346,0.905734
4,without_language,language,92,0.972383,0.021794,0.914925,0.911841
5,without_governance,governance,86,0.967419,0.025738,0.912421,0.908811
6,without_release,release,83,0.964669,0.027875,0.919895,0.917809
7,without_contributor,contributor,88,0.948399,0.026879,0.883781,0.878182
8,only_contributor,contributor,13,0.906680,0.031649,0.846230,0.838555
9,only_event,event,21,0.901607,0.033406,0.836319,0.832448


## Aggregate Feature Ranking

In [13]:
def rank_table(df, score_col, rank_name, ascending=False):
    out = df[["feature", score_col]].copy()
    out[rank_name] = out[score_col].rank(ascending=ascending, method="average")
    return out[["feature", rank_name]]


rank_sources = [
    rank_table(univariate_result, "abs_cohen_d", "rank_univariate", ascending=False),
    rank_table(linear_result, "abs_coef", "rank_linear", ascending=False),
    rank_table(tree_result, "importance", "rank_tree", ascending=False),
    rank_table(permutation_result, "permutation_importance_mean", "rank_permutation", ascending=False),
    rank_table(shap_result, "mean_abs_shap", "rank_shap", ascending=False),
]

aggregate_ranking = pd.DataFrame({"feature": all_model_features})

for source in rank_sources:
    aggregate_ranking = aggregate_ranking.merge(source, on="feature", how="left")

rank_cols = [col for col in aggregate_ranking.columns if col.startswith("rank_")]

for col in rank_cols:
    aggregate_ranking[col] = aggregate_ranking[col].fillna(aggregate_ranking[col].max())

aggregate_ranking["mean_rank"] = aggregate_ranking[rank_cols].mean(axis=1)
aggregate_ranking["family"] = aggregate_ranking["feature"].map(feature_to_family).fillna("other")
aggregate_ranking["source"] = aggregate_ranking["feature"].apply(
    lambda x: "engineered" if x in engineered_features else "raw"
)

aggregate_ranking = aggregate_ranking.sort_values("mean_rank").reset_index(drop=True)
aggregate_ranking.to_csv(OUTPUT_DIR / "aggregate_feature_ranking.csv", index=False)
aggregate_ranking.head(40)


,feature,rank_univariate,rank_linear,rank_tree,rank_permutation,rank_shap,mean_rank,family,source
0,has_IssueCommentEvent,6.0,22.0,1.0,8.0,1.0,7.6,event,raw
1,is_recently_pushed_30d,9.0,33.0,2.0,10.0,2.0,11.2,maintenance,engineered
2,latest_tag_is_stable,49.0,3.0,27.0,2.0,7.0,17.6,release,raw
3,has_PullRequestEvent,10.0,11.0,3.0,73.5,3.0,20.1,event,raw
4,semver_tag_ratio,37.0,37.0,19.0,3.0,10.0,21.2,release,raw
5,has_deployments,45.0,6.0,32.0,4.0,19.0,21.2,release,raw
6,maintainer_activity_score,4.0,64.0,8.0,21.5,14.0,22.3,maintenance,engineered
7,top3_contribution_ratio,11.0,18.0,10.0,65.0,11.0,23.0,contributor,raw
8,contribution_entropy,7.0,9.0,5.0,93.0,5.0,23.8,contributor,raw
9,push_update_consistency,12.0,88.0,4.0,11.0,4.0,23.8,maintenance,engineered


## Candidate Dataset Generation

In [14]:
def remove_highly_correlated_features(X, ranked_features, threshold=0.92):
    corr = X[ranked_features].corr().abs()
    selected = []

    for feature in ranked_features:
        if not selected:
            selected.append(feature)
            continue

        max_corr = corr.loc[feature, selected].max()

        if pd.isna(max_corr) or max_corr < threshold:
            selected.append(feature)

    return selected


def unique_feature_list(features):
    seen = set()
    result = []

    for feature in features:
        if feature in all_model_features and feature not in seen:
            result.append(feature)
            seen.add(feature)

    return result


ranked_features = aggregate_ranking["feature"].tolist()
low_corr_features = remove_highly_correlated_features(
    X_train,
    ranked_features,
    threshold=0.92,
)

strong_ablation_families = (
    ablation_result[ablation_result["setting"].str.startswith("only_")]
    .query("roc_auc_mean >= 0.80")
    ["family"]
    .tolist()
)

family_selected_features = [
    feature for feature in ranked_features
    if feature_to_family.get(feature, "other") in strong_ablation_families
]

core_domain_features = [
    "oss_health_score",
]

feature_sets = {
    "all_features": all_model_features,
    "low_corr_features": low_corr_features,
    "top_15_aggregate": ranked_features[:15],
    "top_20_aggregate": ranked_features[:20],
    "top_25_aggregate": ranked_features[:25],
    "top_30_aggregate": ranked_features[:30],
    "top_40_aggregate": ranked_features[:40],
    "top_50_aggregate": ranked_features[:50],
    "top_30_low_corr": remove_highly_correlated_features(X_train, ranked_features[:45], threshold=0.88)[:30],
    "shap_top_25": shap_result["feature"].head(25).tolist(),
    "tree_top_25": tree_result["feature"].head(25).tolist(),
    "permutation_top_25": permutation_result["feature"].head(25).tolist(),
    "family_ablation_selected": family_selected_features[:40] if family_selected_features else ranked_features[:30],
}

feature_sets = {
    name: unique_feature_list(features)
    for name, features in feature_sets.items()
}

feature_set_summary = pd.DataFrame([
    {
        "dataset_name": name,
        "num_features": len(features),
        "features": features,
    }
    for name, features in feature_sets.items()
])

feature_set_summary.to_csv(OUTPUT_DIR / "candidate_feature_sets.csv", index=False)
feature_set_summary[["dataset_name", "num_features"]]


,dataset_name,num_features
0,all_features,101
1,low_corr_features,77
2,top_15_aggregate,15
3,top_20_aggregate,20
4,top_25_aggregate,25
5,top_30_aggregate,30
6,top_40_aggregate,40
7,top_50_aggregate,50
8,top_30_low_corr,30
9,shap_top_25,25


## Model Zoo

In [15]:
def get_model_zoo():
    return {
        "LogisticRegression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=20000,
                solver="liblinear",
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        "SVC_RBF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(
                kernel="rbf",
                probability=True,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=500,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            )),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=700,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            )),
        ]),
        "GradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingClassifier(
                random_state=RANDOM_STATE,
            )),
        ]),
        "HistGradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingClassifier(
                random_state=RANDOM_STATE,
                max_iter=300,
            )),
        ]),
        "XGBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                n_estimators=500,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            )),
        ]),
        "LightGBM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMClassifier(
                n_estimators=500,
                learning_rate=0.05,
                num_leaves=15,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
                verbose=-1,
            )),
        ]),
        "CatBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostClassifier(
                iterations=500,
                depth=4,
                learning_rate=0.05,
                loss_function="Logloss",
                eval_metric="AUC",
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
            )),
        ]),
    }


## Dataset and Model Search

In [16]:
def evaluate_candidate_datasets(feature_sets, models):
    rows = []

    for dataset_name, features in feature_sets.items():
        X_subset = X_train[features]

        for model_name, model in models.items():
            scores = cross_validate(
                model,
                X_subset,
                y_train,
                cv=cv,
                scoring={
                    "roc_auc": "roc_auc",
                    "f1": "f1",
                    "accuracy": "accuracy",
                    "precision": "precision",
                    "recall": "recall",
                },
                n_jobs=N_JOBS,
                return_train_score=False,
            )

            rows.append({
                "dataset_name": dataset_name,
                "model_name": model_name,
                "num_features": len(features),
                "roc_auc_mean": scores["test_roc_auc"].mean(),
                "roc_auc_std": scores["test_roc_auc"].std(),
                "f1_mean": scores["test_f1"].mean(),
                "accuracy_mean": scores["test_accuracy"].mean(),
                "precision_mean": scores["test_precision"].mean(),
                "recall_mean": scores["test_recall"].mean(),
            })

    return pd.DataFrame(rows).sort_values(
        ["roc_auc_mean", "f1_mean", "accuracy_mean"],
        ascending=False,
    ).reset_index(drop=True)


model_zoo = get_model_zoo()
search_results = evaluate_candidate_datasets(feature_sets, model_zoo)
search_results.to_csv(OUTPUT_DIR / "dataset_model_search_results.csv", index=False)
search_results.head(30)


,dataset_name,model_name,num_features,roc_auc_mean,roc_auc_std,f1_mean,accuracy_mean,precision_mean,recall_mean
0,all_features,LogisticRegression,101,0.992028,0.003288,0.957913,0.957436,0.953376,0.963447
1,low_corr_features,LogisticRegression,77,0.991850,0.004294,0.955023,0.954406,0.947504,0.963447
2,top_40_aggregate,LogisticRegression,40,0.988900,0.005413,0.951575,0.951375,0.952698,0.951326
3,top_50_aggregate,LogisticRegression,50,0.987976,0.004212,0.946207,0.945268,0.943156,0.951326
4,top_30_low_corr,SVC_RBF,30,0.987971,0.006011,0.940774,0.942145,0.964137,0.920833
5,top_40_aggregate,SVC_RBF,40,0.986892,0.008117,0.942334,0.942284,0.942564,0.945265
6,top_50_aggregate,SVC_RBF,50,0.986880,0.007875,0.931134,0.930023,0.925387,0.939205
7,top_30_low_corr,LogisticRegression,30,0.985755,0.006638,0.931383,0.930117,0.926460,0.939205
8,top_50_aggregate,LightGBM,50,0.983207,0.009357,0.906223,0.905688,0.898733,0.914773
9,top_40_aggregate,XGBoost,40,0.981382,0.012205,0.930781,0.930070,0.922932,0.939205


## Holdout Evaluation Before Tuning

In [17]:
best_search_row = search_results.iloc[0]
best_dataset_name = best_search_row["dataset_name"]
best_model_name = best_search_row["model_name"]
best_features = feature_sets[best_dataset_name]

print("Best dataset:", best_dataset_name)
print("Best model:", best_model_name)
print("Number of features:", len(best_features))
print("CV ROC-AUC:", best_search_row["roc_auc_mean"])

baseline_best_model = clone(model_zoo[best_model_name])
baseline_best_model.fit(X_train[best_features], y_train)

y_pred = baseline_best_model.predict(X_test[best_features])
y_prob = baseline_best_model.predict_proba(X_test[best_features])[:, 1]

baseline_holdout_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob),
}

print(classification_report(y_test, y_pred))
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))
print(baseline_holdout_metrics)


Best dataset: all_features
Best model: LogisticRegression
Number of features: 101
CV ROC-AUC: 0.9920282369146005
              precision    recall  f1-score   support

           0       0.93      0.95      0.94        41
           1       0.95      0.93      0.94        42

    accuracy                           0.94        83
   macro avg       0.94      0.94      0.94        83
weighted avg       0.94      0.94      0.94        83

Confusion Matrix
[[39  2]
 [ 3 39]]
{'accuracy': 0.9397590361445783, 'precision': 0.9512195121951219, 'recall': 0.9285714285714286, 'f1': 0.9397590361445783, 'roc_auc': 0.980836236933798}


## Hyperparameter Tuning

In [18]:
def get_search_space(model_name):
    spaces = {
        "LogisticRegression": {
            "model__C": loguniform(1e-3, 1e2),
            "model__penalty": ["l1", "l2"],
        },
        "SVC_RBF": {
            "model__C": loguniform(1e-2, 1e2),
            "model__gamma": loguniform(1e-4, 1e0),
        },
        "RandomForest": {
            "model__n_estimators": randint(300, 1200),
            "model__max_depth": [None, 3, 4, 5, 6, 8, 10],
            "model__min_samples_split": randint(2, 12),
            "model__min_samples_leaf": randint(1, 8),
            "model__max_features": ["sqrt", "log2", None],
        },
        "ExtraTrees": {
            "model__n_estimators": randint(300, 1200),
            "model__max_depth": [None, 3, 4, 5, 6, 8, 10],
            "model__min_samples_split": randint(2, 12),
            "model__min_samples_leaf": randint(1, 8),
            "model__max_features": ["sqrt", "log2", None],
        },
        "GradientBoosting": {
            "model__n_estimators": randint(100, 800),
            "model__learning_rate": loguniform(0.005, 0.2),
            "model__max_depth": randint(1, 5),
            "model__min_samples_leaf": randint(1, 8),
            "model__subsample": uniform(0.65, 0.35),
        },
        "HistGradientBoosting": {
            "model__max_iter": randint(100, 800),
            "model__learning_rate": loguniform(0.005, 0.2),
            "model__max_leaf_nodes": randint(7, 40),
            "model__min_samples_leaf": randint(5, 40),
            "model__l2_regularization": loguniform(1e-4, 1e1),
        },
        "XGBoost": {
            "model__n_estimators": randint(100, 900),
            "model__max_depth": randint(2, 7),
            "model__learning_rate": loguniform(0.005, 0.2),
            "model__subsample": uniform(0.65, 0.35),
            "model__colsample_bytree": uniform(0.65, 0.35),
            "model__min_child_weight": randint(1, 8),
            "model__reg_alpha": loguniform(1e-4, 1e1),
            "model__reg_lambda": loguniform(1e-3, 1e2),
        },
        "LightGBM": {
            "model__n_estimators": randint(100, 900),
            "model__learning_rate": loguniform(0.005, 0.2),
            "model__num_leaves": randint(7, 40),
            "model__max_depth": [-1, 2, 3, 4, 5, 6, 8],
            "model__min_child_samples": randint(5, 50),
            "model__subsample": uniform(0.65, 0.35),
            "model__colsample_bytree": uniform(0.65, 0.35),
            "model__reg_alpha": loguniform(1e-4, 1e1),
            "model__reg_lambda": loguniform(1e-3, 1e2),
        },
        "CatBoost": {
            "model__iterations": randint(100, 900),
            "model__depth": randint(2, 7),
            "model__learning_rate": loguniform(0.005, 0.2),
            "model__l2_leaf_reg": loguniform(1e-2, 1e2),
            "model__bagging_temperature": uniform(0.0, 1.0),
        },
    }

    return spaces[model_name]


n_iter_by_model = {
    "LogisticRegression": 40,
    "SVC_RBF": 40,
    "RandomForest": 50,
    "ExtraTrees": 50,
    "GradientBoosting": 60,
    "HistGradientBoosting": 60,
    "XGBoost": 60,
    "LightGBM": 60,
    "CatBoost": 60,
}

base_model = clone(model_zoo[best_model_name])
search_space = get_search_space(best_model_name)

tuner = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=search_space,
    n_iter=n_iter_by_model.get(best_model_name, 50),
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=1,
    refit=True,
)

tuner.fit(X_train[best_features], y_train)

print("Best model:", best_model_name)
print("Best dataset:", best_dataset_name)
print("Best CV ROC-AUC:", tuner.best_score_)
print("Best params:")
print(tuner.best_params_)

tuning_results = pd.DataFrame(tuner.cv_results_).sort_values(
    "rank_test_score"
).reset_index(drop=True)
tuning_results.to_csv(OUTPUT_DIR / "hyperparameter_tuning_results.csv", index=False)
tuning_results.head(20)


Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best model: LogisticRegression
Best dataset: all_features
Best CV ROC-AUC: 0.9920282369146005
Best params:
{'model__C': np.float64(1.2357483710912178), 'model__penalty': 'l2'}


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__penalty,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.006513,0.000296,0.001516,0.000005,0.655301,l2,"{'model__C': 0.6553013900933984, 'model__penal...",0.991736,0.988062,0.992654,0.998106,0.989583,0.992028,0.003439,1
1,0.006350,0.000145,0.001516,0.000006,0.541441,l2,"{'model__C': 0.5414413211338525, 'model__penal...",0.991736,0.988062,0.992654,0.998106,0.989583,0.992028,0.003439,1
2,0.006710,0.000148,0.001518,0.000008,1.121975,l2,"{'model__C': 1.1219752813215704, 'model__penal...",0.990817,0.988981,0.992654,0.998106,0.989583,0.992028,0.003288,1
3,0.006789,0.000226,0.001533,0.000014,1.235748,l2,"{'model__C': 1.2357483710912178, 'model__penal...",0.990817,0.988981,0.992654,0.998106,0.989583,0.992028,0.003288,1
4,0.006663,0.000173,0.001514,0.000006,0.976113,l2,"{'model__C': 0.9761125443110458, 'model__penal...",0.990817,0.988981,0.992654,0.998106,0.989583,0.992028,0.003288,1
5,0.005796,0.000160,0.001523,0.000009,0.133592,l2,"{'model__C': 0.13359166801651828, 'model__pena...",0.993572,0.983471,0.990817,0.998106,0.991477,0.991489,0.004751,6
6,0.005984,0.000061,0.001531,0.000011,0.178853,l2,"{'model__C': 0.17885301261862016, 'model__pena...",0.993572,0.983471,0.990817,0.998106,0.990530,0.991299,0.004767,7
7,0.006222,0.000201,0.001537,0.000011,0.372539,l2,"{'model__C': 0.37253938395788866, 'model__pena...",0.992654,0.986226,0.989899,0.998106,0.989583,0.991294,0.003970,8
8,0.005534,0.000020,0.001513,0.000004,0.094570,l2,"{'model__C': 0.0945695189734589, 'model__penal...",0.995409,0.981635,0.988981,0.998106,0.991477,0.991121,0.005690,9
9,0.005520,0.000084,0.001514,0.000006,0.090220,l2,"{'model__C': 0.09022004468565618, 'model__pena...",0.995409,0.981635,0.988981,0.998106,0.991477,0.991121,0.005690,9


## Tuned Model Holdout Evaluation

In [19]:
tuned_model = tuner.best_estimator_

y_pred_tuned = tuned_model.predict(X_test[best_features])
y_prob_tuned = tuned_model.predict_proba(X_test[best_features])[:, 1]

tuned_holdout_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_tuned),
    "precision": precision_score(y_test, y_pred_tuned),
    "recall": recall_score(y_test, y_pred_tuned),
    "f1": f1_score(y_test, y_pred_tuned),
    "roc_auc": roc_auc_score(y_test, y_prob_tuned),
}

print(classification_report(y_test, y_pred_tuned))
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred_tuned))
print(tuned_holdout_metrics)

holdout_comparison = pd.DataFrame([
    {"stage": "baseline_best", **baseline_holdout_metrics},
    {"stage": "tuned_best", **tuned_holdout_metrics},
])

holdout_comparison.to_csv(OUTPUT_DIR / "holdout_comparison.csv", index=False)
holdout_comparison


              precision    recall  f1-score   support

           0       0.93      0.95      0.94        41
           1       0.95      0.93      0.94        42

    accuracy                           0.94        83
   macro avg       0.94      0.94      0.94        83
weighted avg       0.94      0.94      0.94        83

Confusion Matrix
[[39  2]
 [ 3 39]]
{'accuracy': 0.9397590361445783, 'precision': 0.9512195121951219, 'recall': 0.9285714285714286, 'f1': 0.9397590361445783, 'roc_auc': 0.9819976771196284}


,stage,accuracy,precision,recall,f1,roc_auc
0,baseline_best,0.939759,0.95122,0.928571,0.939759,0.980836
1,tuned_best,0.939759,0.95122,0.928571,0.939759,0.981998


## Final Training on Full Dataset

In [20]:
final_model = clone(tuner.best_estimator_)
final_model.fit(X_all[best_features], y)

final_cv_scores = cross_validate(
    final_model,
    X_all[best_features],
    y,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "f1": "f1",
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
    },
    n_jobs=N_JOBS,
    return_train_score=False,
)

final_cv_summary = {
    "roc_auc_mean": final_cv_scores["test_roc_auc"].mean(),
    "roc_auc_std": final_cv_scores["test_roc_auc"].std(),
    "f1_mean": final_cv_scores["test_f1"].mean(),
    "accuracy_mean": final_cv_scores["test_accuracy"].mean(),
    "precision_mean": final_cv_scores["test_precision"].mean(),
    "recall_mean": final_cv_scores["test_recall"].mean(),
}

final_cv_summary


{'roc_auc_mean': np.float64(0.9919294070989491),
 'roc_auc_std': np.float64(0.0023104519791599296),
 'f1_mean': np.float64(0.9539797064873555),
 'accuracy_mean': np.float64(0.9536879224213928),
 'precision_mean': np.float64(0.9471490694973396),
 'recall_mean': np.float64(0.9609756097560975)}

## Final Model Interpretation

In [21]:
def get_final_importance(model, X, y):
    model_step = model.named_steps["model"]

    if hasattr(model_step, "feature_importances_"):
        importance = model_step.feature_importances_
        importance_df = pd.DataFrame({
            "feature": X.columns,
            "family": [feature_to_family.get(feature, "other") for feature in X.columns],
            "model_importance": importance,
        }).sort_values("model_importance", ascending=False).reset_index(drop=True)
    elif hasattr(model_step, "coef_"):
        coef = model_step.coef_[0]
        importance_df = pd.DataFrame({
            "feature": X.columns,
            "family": [feature_to_family.get(feature, "other") for feature in X.columns],
            "coef": coef,
            "abs_coef": np.abs(coef),
        }).sort_values("abs_coef", ascending=False).reset_index(drop=True)
    else:
        importance_df = pd.DataFrame({
            "feature": X.columns,
            "family": [feature_to_family.get(feature, "other") for feature in X.columns],
        })

    perm = permutation_importance(
        model,
        X,
        y,
        n_repeats=30,
        random_state=RANDOM_STATE,
        scoring="roc_auc",
        n_jobs=N_JOBS,
    )

    permutation_df = pd.DataFrame({
        "feature": X.columns,
        "family": [feature_to_family.get(feature, "other") for feature in X.columns],
        "permutation_importance_mean": perm.importances_mean,
        "permutation_importance_std": perm.importances_std,
    }).sort_values("permutation_importance_mean", ascending=False).reset_index(drop=True)

    return importance_df, permutation_df


final_importance, final_permutation = get_final_importance(
    final_model,
    X_all[best_features],
    y,
)

final_importance.to_csv(OUTPUT_DIR / "final_model_importance.csv", index=False)
final_permutation.to_csv(OUTPUT_DIR / "final_model_permutation_importance.csv", index=False)

final_importance.head(30)


,feature,family,coef,abs_coef
0,is_monolingual,language,1.742658,1.742658
1,latest_tag_is_stable,release,1.220189,1.220189
2,has_deployments,release,1.194547,1.194547
3,latest_tag_is_prerelease,release,-1.143718,1.143718
4,compiled_ratio,language,1.099371,1.099371
5,has_IssuesEvent,event,0.955275,0.955275
6,repo_age_days,maintenance,0.936895,0.936895
7,subscribers_count,popularity,0.842057,0.842057
8,has_IssueCommentEvent,event,0.840174,0.840174
9,has_PullRequestEvent,event,0.829598,0.829598


In [22]:
try:
    imputer = final_model.named_steps.get("imputer")
    model_step = final_model.named_steps["model"]
    X_for_shap = X_all[best_features]

    if imputer is not None:
        X_for_shap_values = imputer.transform(X_for_shap)
    else:
        X_for_shap_values = X_for_shap.values

    explainer = shap.TreeExplainer(model_step)
    shap_values = explainer.shap_values(X_for_shap_values)

    if isinstance(shap_values, list):
        final_shap_positive = np.asarray(shap_values[1])
    else:
        shap_array = np.asarray(shap_values)

        if shap_array.ndim == 3 and shap_array.shape[2] == 2:
            final_shap_positive = shap_array[:, :, 1]
        elif shap_array.ndim == 3 and shap_array.shape[0] == 2:
            final_shap_positive = shap_array[1, :, :]
        elif shap_array.ndim == 2:
            final_shap_positive = shap_array
        else:
            raise ValueError(f"Unexpected SHAP shape: {shap_array.shape}")

    final_shap_importance = pd.DataFrame({
        "feature": best_features,
        "family": [feature_to_family.get(feature, "other") for feature in best_features],
        "mean_abs_shap": np.abs(final_shap_positive).mean(axis=0).ravel(),
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

except Exception as exc:
    final_shap_importance = pd.DataFrame()
    print("SHAP skipped:", repr(exc))

if not final_shap_importance.empty:
    final_shap_importance.to_csv(OUTPUT_DIR / "final_model_shap_importance.csv", index=False)

final_shap_importance.head(30)


SHAP skipped: InvalidModelError("Model type not yet supported by TreeExplainer: <class 'sklearn.linear_model._logistic.LogisticRegression'>")


""


## Save Backend-ready Artifacts

In [23]:
model_path = MODEL_DIR / "oss_health_best_model.joblib"
feature_path = MODEL_DIR / "oss_health_best_features.json"
metadata_path = MODEL_DIR / "oss_health_model_metadata.json"
training_dataset_path = OUTPUT_DIR / "final_training_dataset.csv"

joblib.dump(final_model, model_path)

with open(feature_path, "w", encoding="utf-8") as f:
    json.dump(best_features, f, ensure_ascii=False, indent=2)

metadata = {
    "target_col": TARGET_COL,
    "data_path": str(DATA_PATH),
    "best_dataset_name": best_dataset_name,
    "best_model_name": best_model_name,
    "num_features": len(best_features),
    "features": best_features,
    "baseline_holdout_metrics": baseline_holdout_metrics,
    "tuned_holdout_metrics": tuned_holdout_metrics,
    "final_cv_summary": final_cv_summary,
    "best_cv_score_before_tuning": float(best_search_row["roc_auc_mean"]),
    "best_tuning_cv_score": float(tuner.best_score_),
    "best_params": tuner.best_params_,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

modeling_df[best_features + [TARGET_COL]].to_csv(training_dataset_path, index=False)

print("Saved model:", model_path)
print("Saved features:", feature_path)
print("Saved metadata:", metadata_path)
print("Saved final training dataset:", training_dataset_path)


Saved model: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/models/oss_health_best_model.joblib
Saved features: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/models/oss_health_best_features.json
Saved metadata: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/models/oss_health_model_metadata.json
Saved final training dataset: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/outputs/2_model/final_training_dataset.csv


## Backend Inference Example

In [24]:
loaded_model = joblib.load(model_path)

with open(feature_path, "r", encoding="utf-8") as f:
    loaded_features = json.load(f)

sample_X = X_all[loaded_features].head(5)
sample_prob = loaded_model.predict_proba(sample_X)[:, 1]
sample_pred = loaded_model.predict(sample_X)

pd.DataFrame({
    "predicted_label": sample_pred,
    "healthy_probability": sample_prob,
})


,predicted_label,healthy_probability
0,1,0.999947
1,0,0.000024
2,1,0.999988
3,1,0.999806
4,1,0.999889
